In [2]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

from src.evaluation import ranked_probability_score, rps_from_lambdas

In [4]:
# Load the cleaned dataset
results = pd.read_parquet('../data/interim/results_clean.parquet')

# Reproduce the train/eval/WC2026 split (same as previous notebooks)
model_df = results[results['date'].dt.year >= 1970].reset_index(drop=True)
wc2026_mask = (model_df['tournament'] == 'FIFA World Cup') & (model_df['date'].dt.year == 2026)
train_df = model_df[model_df['date'] < '2024-01-01']
eval_df = model_df[(model_df['date'] >= '2024-01-01') & ~wc2026_mask]
wc2026_df = model_df[wc2026_mask]

print(f'Train: {len(train_df)} | Eval: {len(eval_df)} | WC2026: {len(wc2026_df)}')

Train: 38900 | Eval: 2543 | WC2026: 79


In [10]:
# Load all deployed models from 05_ensemble (trained on train + eval set & tuned)
model_dir = Path('../models')
cb_tuned = joblib.load(model_dir / 'cb_deployed.joblib')
lgb_tuned = joblib.load(model_dir / 'lgb_deployed.joblib')
xgb_tuned = joblib.load(model_dir / 'xgb_deployed.joblib')
hgb_tuned = joblib.load(model_dir / 'hgb_deployed.joblib')
rf_tuned = joblib.load(model_dir / 'rf_deployed.joblib')
glm_tuned = joblib.load(model_dir / 'glm_deployed.joblib')

print('6 tuned learners loaded from disk')

# Load feature-engineered model_df from 05_ensemble 
model_df = pd.read_parquet('../data/interim/model_df_featured.parquet')
print('\nmodel_df loaded from disk')

6 tuned learners loaded from disk

model_df loaded from disk


In [15]:
def build_team_state(matches, k=30, base_rating=1500, form_window=5):
    """Walk matches chronologically and snapshot each team's state at the end.

    Returns dict: {team: {'elo', 'recent_scored', 'recent_conceded', 'last_date'}}.
    Used to score hypothetical 2026 fixtures where feature rows don't yet exist.
    """
    matches = matches.sort_values('date').reset_index(drop=True)

    ratings = {}
    recent_scored = {}
    recent_conceded = {}
    last_date = {}

    for _, row in matches.iterrows():
        home_team = row['home_team']
        away_team = row['away_team']
        home_goals = row['home_score']
        away_goals = row['away_score']

        # Elo update (same logic as add_elo_features)
        elo_home = ratings.get(home_team, base_rating)
        elo_away = ratings.get(away_team, base_rating)

        expected_home = 1 / (1 + 10 ** ((elo_away - elo_home) / 400))

        if home_goals > away_goals:
            actual_home = 1
        elif home_goals == away_goals:
            actual_home = 0.5
        else:
            actual_home = 0

        change = k * (actual_home - expected_home)
        ratings[home_team] = elo_home + change
        ratings[away_team] = elo_away - change

        # Rolling form - keep the last 'form_window' matches per team
        recent_scored.setdefault(home_team, []).append(home_goals)
        recent_conceded.setdefault(home_team, []).append(away_goals)
        recent_scored.setdefault(away_team, []).append(away_goals)
        recent_conceded.setdefault(away_team, []).append(home_goals)

        recent_scored[home_team] = recent_scored[home_team][-form_window:]
        recent_conceded[home_team] = recent_conceded[home_team][-form_window:]
        recent_scored[away_team] = recent_scored[away_team][-form_window:]
        recent_conceded[away_team] = recent_conceded[away_team][-form_window:]

        last_date[home_team] = row['date']
        last_date[away_team] = row['date']

    return {
        team: {
            'elo': ratings[team],
            'recent_scored': recent_scored[team],
            'recent_conceded': recent_conceded[team],
            'last_date': last_date[team]
        }
        for team in ratings 
    }

pre_wc_df = model_df[~wc2026_mask]
team_state = build_team_state(pre_wc_df)

# Sanity check: top 10 by elo 
top_teams = sorted(team_state.items(), key=lambda kv: kv[1]['elo'], reverse=True)[:10]
for team, state in top_teams:
    print(f"{team:20s} Elo {state['elo']:7.1f}  last {state['last_date'].date()}")

Argentina            Elo  2042.8  last 2026-06-09
Spain                Elo  2038.9  last 2026-06-08
France               Elo  1981.5  last 2026-06-08
Portugal             Elo  1948.1  last 2026-06-10
Brazil               Elo  1944.0  last 2026-06-06
England              Elo  1938.0  last 2026-06-10
Germany              Elo  1937.1  last 2026-06-06
Colombia             Elo  1926.5  last 2026-06-07
Netherlands          Elo  1911.1  last 2026-06-08
Japan                Elo  1908.6  last 2026-05-31
